# Google AI API (Gemini) 실습

이 노트북은 Google AI Studio에서 발급한 API 키로 Gemini API를 호출하는 기본 실습 자료입니다.

> API 키는 절대 노트북에 직접 입력하거나 GitHub에 업로드하지 마세요. `.env` 파일로 관리합니다.

## 1. Gemini API란?

Gemini API는 애플리케이션에서 Gemini 모델을 호출하기 위한 개발자용 인터페이스입니다. 텍스트 생성뿐 아니라 이미지·PDF·오디오·동영상 입력 이해, 구조화된 JSON 출력, 함수 호출, Google 검색 기반 답변 등 멀티모달 AI 기능을 만들 수 있습니다.

- **Google AI Studio**: API 키 발급, 프롬프트 시험, 사용량 확인
- **Gemini Developer API**: 개인·교육·프로토타입 및 일반 애플리케이션 개발에 적합
- **Vertex AI**: IAM·리전·조직 정책 등이 필요한 GCP 기반 기업 운영 환경에 적합

## 2. API 키 발급

1. [Google AI Studio API Keys](https://aistudio.google.com/api-keys)에 접속합니다.
2. API 키를 발급합니다.
3. 발급한 키는 다음 단계의 `.env` 파일에 저장합니다.

## 3. 라이브러리 설치

터미널에서 프로젝트를 사용하는 경우 아래 명령을 실행합니다.

In [ ]:
# 터미널에서 실행
# uv add google-genai python-dotenv pydantic

# Jupyter Notebook 안에서 설치할 때는 아래 줄의 주석을 해제하세요.
# %pip install -U google-genai python-dotenv pydantic

## 4. `.env` 파일에 API 키 저장

프로젝트 폴더에 `.env` 파일을 만들고 다음처럼 작성합니다. 실제 API 키는 외부에 공유하지 않습니다.

In [ ]:
# .env 파일에 Gemini 환경변수를 설정합니다.
# GEMINI_API_KEY='YOUR_GEMINI_API_KEY'
# GEMINI_DEFAULT_MODEL='gemini-3.5-flash-lite'

In [ ]:
from dotenv import load_dotenv
import os
from google import genai

load_dotenv(override=True)
gemini_api_key = os.getenv('GEMINI_API_KEY')
gemini_default_model = os.getenv('GEMINI_DEFAULT_MODEL', 'gemini-3.5-flash-lite')

if not gemini_api_key:
    raise ValueError('GEMINI_API_KEY를 .env 파일에 설정하세요.')

# GEMINI_API_KEY 환경 변수를 SDK가 자동으로 읽습니다.
client = genai.Client()
print(f'기본 모델: {gemini_default_model}')

## 5. 사용 가능한 모델 확인

계정과 지역에 따라 사용 가능한 모델이 다를 수 있으므로 먼저 모델 목록을 확인합니다.

In [ ]:
print('=== 사용 가능 모델 ===')
for model in client.models.list():
    print(model.name)

### 모델 선택 가이드

| 목적 | 권장 모델 | 이유 |
| --- | --- | --- |
| 일반 챗봇·문서 요약·이미지/PDF 분석·코딩 실습 | `gemini-3.8-flash` | 최신 안정 Flash 모델, 멀티모달·도구·구조화 출력 지원 |
| 번역·분류·추출·대량 처리 | `gemini-3.5-flash-lite` | 빠르고 비용 효율적인 대용량 처리용 |
| 실시간 음성 대화 | `gemini-3.8-live` | 저지연 음성-음성 Live API |
| 음성 → 텍스트 | `gemini-3.5-transcribe` | 언어 감지·화자 분리·단어 타임스탬프 지원 |
| 텍스트 → 음성 | `gemini-3.8-flash-tts` | 고품질·표현력 중심 음성 생성 |
| 이미지 생성·편집 | `gemini-3.1-flash-image` | 빠른 이미지 생성·편집 |
| RAG 임베딩 | `gemini-embedding-001` | 텍스트 의미 벡터 생성 |

> 위 모델 이름은 예시입니다. 반드시 앞 단계에서 출력된 사용 가능 모델 목록을 확인해 선택하세요.

## 6. 첫 번째 Gemini 호출

`generate_content()`에 모델, 질문(`contents`), 생성 옵션(`config`)을 전달합니다.

In [ ]:
# 3. 모델 객체 생성 없이 직접 generate_content 호출 (특정 모델 지정)
response = client.models.generate_content(
    model=gemini_default_model,  
    contents="안녕하세요! 삼성전자 뉴스 감성분석 도와주세요."
)

# 4. 응답 출력
print(response.text)

In [ ]:
from google.genai import types

response = client.models.generate_content(
    model=gemini_default_model,
    contents='AI 에이전트를 초보자도 이해할 수 있게 정확히 3문장으로 설명해줘.',
    config=types.GenerateContentConfig(
        temperature=0.3,
        max_output_tokens=1024,
    ),
)

print(response.text)

| 항목 | 설명 |
| --- | --- |
| `model` | 호출할 모델 ID |
| `contents` | 문자열 또는 대화·멀티모달 입력 |
| `config` | 생성 옵션, 시스템 지시, 도구 등을 설정 |
| `response.text` | 모델이 생성한 텍스트 편의 속성 |

### 자주 쓰는 생성 옵션

| 옵션 | 설명 | 권장 사용 |
| --- | --- | --- |
| `temperature` | 답변의 다양성 조절 | 분류·추출: 0~0.3, 아이디어: 0.7 전후 |
| `max_output_tokens` | 최대 출력 길이 | 짧은 응답일수록 비용·지연 감소 |
| `stop_sequences` | 특정 문자열에서 생성 중지 | 형식이 엄격한 출력에 사용 |
| `system_instruction` | 모델의 역할·규칙·답변 언어 | 모든 챗봇에 권장 |

## 7. System Instruction과 역할(Role)

Gemini에서는 모델 전체의 행동 규칙을 `system_instruction`에 둡니다. 대화 내용의 역할은 `user`와 `model`을 사용합니다. OpenAI의 `assistant` 대신 Gemini에서는 `model`을 사용한다는 점에 주의하세요.

In [ ]:
response = client.models.generate_content(
    model=gemini_default_model,
    contents='봄에 대해 알려줘',
    config=types.GenerateContentConfig(
        system_instruction='너는 자바 개발자로 스프링부트 프레임워크를 사용해 웹 콘텐츠를 개발하는 전문가야.',
        temperature=0.2,
        max_output_tokens=2000,
    ),
)
print(response.text)

| 목적 | OpenAI 표현 | Gemini (`google-genai`) 표현 |
| --- | --- | --- |
| 고정 지침 | `system` 메시지 | `system_instruction` |
| 사용자 입력 | `user` | `user` |
| 모델의 이전 답변 | `assistant` | `model` |

## 8. 단일 턴 / 멀티 턴 챗봇

`client.chats.create()`로 만든 `chat` 객체는 이전 대화를 자동으로 기억하므로 여러 번 `send_message()`를 호출할 수 있습니다.

In [ ]:
chat = client.chats.create(
    model=gemini_default_model,
    config=types.GenerateContentConfig(
        system_instruction='당신은 친절한 파이썬 강사입니다. 모든 답변은 한국어로 합니다.',
        temperature=0.3,
        max_output_tokens=500,
    ),
)

question = input('질문: ')
response = chat.send_message(question)
print('Gemini:', response.text)

## 9. 차트 이미지 분석

텍스트와 이미지 바이트 데이터를 함께 전달하면 이미지 내용을 분석할 수 있습니다. 아래 예제를 실행하려면 `./images/chart.png` 경로에 분석할 이미지를 넣으세요.

In [ ]:
with open('./images/chart.png', 'rb') as file:
    image_data = file.read()

response = client.models.generate_content(
    model=gemini_default_model,
    contents=[
        '이 차트 결과를 한국어로 자세히 설명해줘. 긍정/부정 비율과 주요 키워드를 분석해줘.',
        {
            'inline_data': {
                'data': image_data,
                'mime_type': 'image/png',
            }
        },
    ],
)
print(response.text)

### `mime_type` 참고

| 파일 종류 | 확장자 | `mime_type` 값 |
| --- | --- | --- |
| PNG 이미지 | `.png` | `image/png` |
| JPEG 이미지 | `.jpg`, `.jpeg` | `image/jpeg` |
| 웹 이미지 | `.webp` | `image/webp` |
| PDF 문서 | `.pdf` | `application/pdf` |
| 텍스트 파일 | `.txt` | `text/plain` |
| CSV 파일 | `.csv` | `text/csv` |

## 10. 구조화된 출력(JSON)

자연어 결과를 화면·DB·API에 연결하려면 프롬프트만으로 JSON 형식을 요청하는 대신 `response_schema`를 지정하는 편이 안전합니다.

In [ ]:
from pydantic import BaseModel, Field

class SentimentResult(BaseModel):
    label: str = Field(description='긍정, 부정, 중립 중 하나')
    confidence: int = Field(ge=0, le=100)
    reason: str

review = '배송은 빨랐지만 제품 마감이 아쉬워요.'

response = client.models.generate_content(
    model=gemini_default_model,
    contents=review,
    config=types.GenerateContentConfig(
        system_instruction='상품 리뷰의 감정을 분석하세요.',
        response_mime_type='application/json',
        response_schema=SentimentResult,
    ),
)

result = response.parsed
print(result.label, result.confidence, result.reason)

## 11. 최신 정보: Google Search Grounding

Google Search 도구를 연결하면 모델의 학습 지식만으로 답하기보다 웹 검색 결과를 근거로 최신 정보를 답하게 할 수 있습니다.

In [ ]:
response = client.models.generate_content(
    model=gemini_default_model,
    contents='오늘 발표된 Gemini API 관련 주요 변경 사항을 간단히 정리하고 출처도 알려줘.',
    config=types.GenerateContentConfig(
        tools=[types.Tool(google_search=types.GoogleSearch())]
    ),
)

print(response.text)

> 검색 Grounding은 항상 최신·정확함을 보장하지 않습니다. 중요한 의사결정에는 사람이 출처를 직접 확인해야 합니다. 검색·지도 Grounding의 사용 가능 여부와 무료 한도는 모델 및 요금제에 따라 다를 수 있습니다.

## 12. 함수 호출(Function Calling)

모델은 직접 날씨·DB·사내 시스템을 실행하지 않습니다. 대신 어떤 함수를 어떤 인자로 호출할지 요청하고, 프로그램이 실제 함수를 실행한 결과를 모델에 돌려줍니다.

In [ ]:
def get_weather(city: str) -> dict:
    """도시의 현재 날씨를 반환한다. 실제 서비스에서는 날씨 API를 호출한다."""
    sample = {'서울': {'temperature': 21, 'condition': '맑음'}}
    return sample.get(city, {'error': '지원하지 않는 도시입니다.'})

chat = client.chats.create(
    model=gemini_default_model,
    config=types.GenerateContentConfig(tools=[get_weather]),
)

response = chat.send_message('서울 날씨를 알려줘.')
print(response.text)

## 13. 모델별 비용 요약

| 모델 | 입력 1M tokens | 출력 1M tokens | 주 용도 |
| --- | ---: | ---: | --- |
| `gemini-3.8-flash` | $0.75* | $3.75* | 기본 챗봇, 멀티모달, 코딩·에이전트 |
| `gemini-3.5-flash-lite` | $0.30 | $2.50 | 번역·분류·추출 등 대량 처리 |
| `gemini-3.5-flash` | $1.50 | $9.00 | 이전 세대 Flash, 신규 기본값으로는 비추천 |

> 가격·모델 제공 여부는 변경될 수 있으므로 배포 전 [Google AI for Developers 가격 문서](https://ai.google.dev/gemini-api/docs/pricing)를 확인하세요.